# Age & Gender Prediction — Improved Model (Memory Efficient)

## Key Improvements Over Original
- **Transfer Learning** with MobileNetV2 (ImageNet pre-trained backbone)
- **Multi-Task Learning** — shared feature extractor with separate age/gender heads
- **Data Augmentation** — rotation, flips, brightness, zoom, shifts
- **GlobalAveragePooling2D** instead of Flatten (reduces overfitting)
- **Learning Rate Scheduling** with ReduceLROnPlateau
- **Early Stopping** to prevent overfitting
- **Class Weights** for imbalanced age groups
- **Consistent 224x224 input size** throughout
- **BatchNormalization + Dropout** in classification heads
- **Memory Efficient** — uses custom generator (loads images on-the-fly, NO OOM!)


In [ ]:
# ===============================
# IMPORT LIBRARIES
# ===============================

import os
import glob
import gc
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten,
    Dense, Dropout, BatchNormalization,
    GlobalAveragePooling2D, Input
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical, Sequence
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    mean_absolute_error
)
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))


In [ ]:
# ===============================
# PARAMETERS
# ===============================

IMG_SIZE: int = 224           # Spatial input dimension for MobileNetV2.
BATCH_SIZE: int = 64          # Number of samples per gradient update.
EPOCHS: int = 50              # Maximum training epochs per phase.
LEARNING_RATE: float = 1e-4   # Initial learning rate for Phase 1.
TEST_SPLIT: float = 0.15      # Fraction of data reserved for final evaluation.
VAL_SPLIT: float = 0.15       # Fraction of remaining data for validation.
RANDOM_STATE: int = 42        # Seed for reproducible splits.
NUM_WORKERS: int = 2          # Parallel workers for data loading.


In [ ]:
# ===============================
# LOAD UTKFACE DATASET
# ===============================

# Update this path to your UTKFace folder location
data_path: str = "/kaggle/input/datasets/nickstarzayswal/utkdata-face/UTKFace"

image_paths: list[str] = glob.glob(os.path.join(data_path, "*.jpg"))

if len(image_paths) == 0:
    print("No images found. Check your folder path and ensure images are in .jpg format.")
else:
    print(f"Found {len(image_paths)} images. Dataset loaded successfully!")


In [ ]:
# ===============================
# PARSE FILENAMES (NO IMAGE LOADING)
# ===============================
#
# This cell ONLY parses filenames to extract age and gender labels.
# We do NOT load images into memory here — that happens later
# in the custom generator, one batch at a time.
#
# This prevents the Out Of Memory (OOM) crash on Kaggle!

valid_paths: list[str] = []
ages: list[int] = []
genders: list[int] = []
skipped: int = 0

for img_path in tqdm(image_paths, desc="Parsing filenames"):
    filename = os.path.basename(img_path)

    try:
        age = int(filename.split("_")[0])
        gender = int(filename.split("_")[1])
    except (ValueError, IndexError):
        skipped += 1
        continue

    # Discard biologically implausible ages
    if age < 0 or age > 116:
        skipped += 1
        continue

    valid_paths.append(img_path)
    ages.append(age)
    genders.append(gender)

print(f"\nValid images: {len(valid_paths)}")
print(f"Skipped: {skipped}")
print(f"Age range: {min(ages)} - {max(ages)}")
print(f"Gender: Male={genders.count(0)}, Female={genders.count(1)}")


In [ ]:
# ===============================
# EXPLORE AGE DISTRIBUTION
# ===============================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(ages, bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Raw Age Distribution")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Count")

gender_counts = [genders.count(0), genders.count(1)]
axes[1].pie(gender_counts, labels=["Male (0)", "Female (1)"],
            autopct="%1.1f%%", colors=["#4c72b0", "#dd8452"], startangle=90)
axes[1].set_title("Gender Distribution")

plt.tight_layout()
plt.show()


In [ ]:
# ===============================
# AGE BINNING (IMPROVED BALANCED BINS)
# ===============================
#
# The original bins [0,2,6,13,20,32,43,53,100] created very
# imbalanced classes. These new bins are designed for more
# balanced class sizes:
#
#   0:   (0, 3]     infants/toddlers
#   1:   (3, 8]     young children
#   2:   (8, 15]    older children / early teens
#   3:   (15, 22]   late teens / young adults
#   4:   (22, 30]   adults
#   5:   (30, 40]   adults
#   6:   (40, 50]   middle-aged
#   7:   (50, 60]   mature adults
#   8:   (60, 116]  seniors

age_bins: list[int] = [0, 3, 8, 15, 22, 30, 40, 50, 60, 116]
age_labels_list: list[str] = [
    "0-3",
    "4-8",
    "9-15",
    "16-22",
    "23-30",
    "31-40",
    "41-50",
    "51-60",
    "61+",
]

# np.digitize returns 1-based indices; subtract 1 to get 0-based
y_age_group: np.ndarray = np.digitize(ages, age_bins) - 1
NUM_AGE_CLASSES: int = len(age_labels_list)

y_gender: np.ndarray = np.array(genders, dtype=np.int32)
y_age_cat: np.ndarray = to_categorical(y_age_group, num_classes=NUM_AGE_CLASSES)

print(f"Number of age classes: {NUM_AGE_CLASSES}")
print("Age group distribution:")
for i, label in enumerate(age_labels_list):
    count = np.sum(y_age_group == i)
    pct = count / len(y_age_group) * 100
    print(f"  Class {i} ({label:>6}): {count:>6} images ({pct:.1f}%)")


In [ ]:
# ===============================
# CUSTOM DATA GENERATOR (Memory Efficient)
# ===============================
#
# This generator loads images from disk on-the-fly in batches.
# Only BATCH_SIZE images are in memory at any time.
# This prevents the Out Of Memory crash!


class FaceDataGenerator(Sequence):
    """Keras Sequence that lazily loads face images in batches from disk.

    Instead of pre-loading the entire UTKFace dataset into RAM (~14 GB),
    this generator reads one batch of images from disk at each step.
    Optionally applies real-time data augmentation to training batches.

    Attributes:
        image_paths: Numpy array of absolute file paths to JPG images.
        y_gender: 1-D int32 array of gender labels (0=Male, 1=Female).
        y_age_cat: 2-D float32 array of one-hot encoded age group labels.
        batch_size: Number of images yielded per __getitem__ call.
        img_size: Spatial size to which each image is resized.
        augment: Whether to apply random augmentation transforms.
        shuffle: Whether to reshuffle indices at the end of each epoch.
        indices: Current ordering of sample indices (permuted when shuffle=True).
        datagen: ImageDataGenerator instance (or None if augment=False).
    """

    def __init__(
        self,
        image_paths: list[str],
        y_gender: np.ndarray,
        y_age_cat: np.ndarray,
        batch_size: int = 64,
        img_size: int = 224,
        augment: bool = False,
        shuffle: bool = True,
    ) -> None:
        """Initialise the data generator.

        Args:
            image_paths: List of absolute paths to face images on disk.
            y_gender: 1-D numpy array of binary gender labels.
            y_age_cat: 2-D numpy array of one-hot age group labels.
            batch_size: Number of samples per batch. Defaults to 64.
            img_size: Target spatial dimension for resizing. Defaults to 224.
            augment: If True, apply random augmentation at each step.
                Defaults to False.
            shuffle: If True, reshuffle indices every epoch.
                Defaults to True.
        """
        self.image_paths: np.ndarray = np.array(image_paths)
        self.y_gender: np.ndarray = np.array(y_gender)
        self.y_age_cat: np.ndarray = np.array(y_age_cat)
        self.batch_size: int = batch_size
        self.img_size: int = img_size
        self.augment: bool = augment
        self.shuffle: bool = shuffle
        self.indices: np.ndarray = np.arange(len(self.image_paths))

        # Configure real-time augmentation pipeline for training batches
        if augment:
            self.datagen: ImageDataGenerator | None = ImageDataGenerator(
                rotation_range=15,
                width_shift_range=0.1,
                height_shift_range=0.1,
                horizontal_flip=True,
                brightness_range=[0.85, 1.15],
                zoom_range=0.1,
                shear_range=0.05,
                fill_mode="nearest",
            )
        else:
            self.datagen: ImageDataGenerator | None = None

        if self.shuffle:
            np.random.shuffle(self.indices)

    def __len__(self) -> int:
        """Return the number of batches per epoch.

        The last batch may be smaller than batch_size if the
        dataset size is not perfectly divisible.

        Returns:
            Integer count of batches in one full pass over the data.
        """
        return int(np.ceil(len(self.image_paths) / self.batch_size))

    def __getitem__(
        self,
        idx: int,
    ) -> tuple[np.ndarray, dict[str, np.ndarray]]:
        """Generate one batch of images and labels.

        Reads images from disk for the requested batch index,
        normalises pixel values to [0, 1], optionally applies
        augmentation, and returns images paired with a multi-output
        label dictionary.

        Args:
            idx: Zero-based batch index (0 <= idx < len(self)).

        Returns:
            A tuple of:
                - images: float32 array of shape (B, img_size, img_size, 3).
                - labels: dict with keys "gender_output" (B,) and
                  "age_output" (B, num_age_classes).

        Raises:
            FileNotFoundError: If any image path cannot be read by cv2.
        """
        batch_indices = self.indices[
            idx * self.batch_size : (idx + 1) * self.batch_size
        ]

        batch_images: list[np.ndarray] = []
        for i in batch_indices:
            img = cv2.imread(self.image_paths[i])
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (self.img_size, self.img_size))
            img = img / 255.0
            batch_images.append(img)

        batch_images: np.ndarray = np.array(batch_images, dtype=np.float32)

        # Apply per-sample random augmentation (label-invariant transforms)
        if self.augment and self.datagen is not None:
            for i in range(len(batch_images)):
                batch_images[i] = self.datagen.random_transform(batch_images[i])

        batch_gender: np.ndarray = self.y_gender[batch_indices]
        batch_age: np.ndarray = self.y_age_cat[batch_indices]

        return batch_images, {
            "gender_output": batch_gender,
            "age_output": batch_age
        }

    def on_epoch_end(self) -> None:
        """Shuffle sample indices at the end of each epoch.

        Called automatically by Keras after every epoch. Only
        performs a shuffle when self.shuffle is True.
        """
        if self.shuffle:
            np.random.shuffle(self.indices)


print("Custom FaceDataGenerator class defined!")
print("This loads images on-the-fly — NO OOM crashes!")


In [ ]:
# ===============================
# TRAIN / VALIDATION / TEST SPLIT
# ===============================
#
# We split the file PATHS (not images), so no pixel data is
# loaded into memory at this point.

# First split: 15 % held out for final test evaluation
paths_train_val, paths_test, y_gender_train_val, y_gender_test, \
y_age_train_val, y_age_test = train_test_split(
    valid_paths, y_gender, y_age_cat,
    test_size=TEST_SPLIT, random_state=RANDOM_STATE, stratify=y_age_group
)

y_age_group_train_val = np.argmax(y_age_train_val, axis=1)

# Second split: 15 % of remainder for validation (~12.75 % of total)
paths_train, paths_val, y_gender_train, y_gender_val, \
y_age_train, y_age_val = train_test_split(
    paths_train_val, y_gender_train_val, y_age_train_val,
    test_size=VAL_SPLIT / (1 - TEST_SPLIT),
    random_state=RANDOM_STATE, stratify=y_age_group_train_val
)

print(f"Training set:   {len(paths_train)} images")
print(f"Validation set: {len(paths_val)} images")
print(f"Test set:       {len(paths_test)} images")

# Free intermediate variables to reclaim RAM
del paths_train_val, y_gender_train_val, y_age_train_val, y_age_group_train_val
gc.collect()


In [ ]:
# ===============================
# CREATE DATA GENERATORS
# ===============================

train_gen = FaceDataGenerator(
    paths_train, y_gender_train, y_age_train,
    batch_size=BATCH_SIZE, img_size=IMG_SIZE,
    augment=True, shuffle=True
)

val_gen = FaceDataGenerator(
    paths_val, y_gender_val, y_age_val,
    batch_size=BATCH_SIZE, img_size=IMG_SIZE,
    augment=False, shuffle=False
)

print(f"Training batches per epoch:   {len(train_gen)}")
print(f"Validation batches per epoch: {len(val_gen)}")

# Quick smoke-test: load one batch and verify shapes
test_imgs, test_labels = train_gen[0]
print(f"\nBatch image shape: {test_imgs.shape}")
print(f"Batch gender labels: {test_labels['gender_output'].shape}")
print(f"Batch age labels:   {test_labels['age_output'].shape}")


In [ ]:
# ===============================
# COMPUTE CLASS WEIGHTS FOR AGE GROUPS
# ===============================
#
# Balanced class weights compensate for unequal group sizes
# so the loss function does not bias toward majority classes.

y_age_group_train = np.argmax(y_age_train, axis=1)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_AGE_CLASSES),
    y=y_age_group_train
)
age_class_weights: dict[int, float] = dict(enumerate(class_weights_array))

print("Age class weights:")
for i, w in age_class_weights.items():
    print(f"  Class {i} ({age_labels_list[i]}): weight = {w:.3f}")


In [ ]:
# ===============================
# BUILD MULTI-TASK MODEL (IMPROVED)
# ===============================


def build_improved_model(num_age_classes: int) -> tuple[Model, Model]:
    """Build a multi-task age/gender model with a MobileNetV2 backbone.

    Constructs a Keras Model with a shared MobileNetV2 feature extractor
    and two task-specific classification heads:
      - Gender head: Dense -> BN -> Dropout -> Dense -> BN -> Dropout -> Sigmoid
      - Age head:    Dense -> BN -> Dropout -> Dense -> BN -> Dropout -> Softmax

    The backbone is initialised with ImageNet weights but starts in a
    frozen (non-trainable) state. Callers should unfreeze selected
    layers before Phase 2 fine-tuning.

    Args:
        num_age_classes: Number of age-group output classes.
            Must match len(age_labels_list) used during label encoding.

    Returns:
        A tuple of:
            - model: The compiled multi-output Keras Model with outputs
              named "gender_output" (sigmoid, shape (None, 1)) and
              "age_output" (softmax, shape (None, num_age_classes)).
            - backbone: The MobileNetV2 sub-model reference. Keep this
              to selectively unfreeze layers for fine-tuning.

    Raises:
        ValueError: If num_age_classes is less than 1.
    """
    if num_age_classes < 1:
        raise ValueError(f"num_age_classes must be >= 1, got {num_age_classes}")

    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input")

    backbone = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_tensor=inputs,
        alpha=1.0
    )

    # Freeze backbone; only classification heads are trainable in Phase 1
    backbone.trainable = False

    x = backbone.output
    x = GlobalAveragePooling2D(name="gap")(x)
    x = BatchNormalization(name="bn_shared")(x)
    x = Dropout(0.3, name="dropout_shared")(x)

    # -- Gender head (binary classification) --
    g = Dense(128, activation="relu", name="gender_dense1")(x)
    g = BatchNormalization(name="gender_bn1")(g)
    g = Dropout(0.3, name="gender_dropout1")(g)
    g = Dense(64, activation="relu", name="gender_dense2")(g)
    g = BatchNormalization(name="gender_bn2")(g)
    g = Dropout(0.2, name="gender_dropout2")(g)
    gender_output = Dense(1, activation="sigmoid", name="gender_output")(g)

    # -- Age head (multi-class classification) --
    a = Dense(256, activation="relu", name="age_dense1")(x)
    a = BatchNormalization(name="age_bn1")(a)
    a = Dropout(0.3, name="age_dropout1")(a)
    a = Dense(128, activation="relu", name="age_dense2")(a)
    a = BatchNormalization(name="age_bn2")(a)
    a = Dropout(0.2, name="age_dropout2")(a)
    age_output = Dense(num_age_classes, activation="softmax", name="age_output")(a)

    model = Model(inputs=inputs, outputs=[gender_output, age_output],
                  name="age_gender_model")

    return model, backbone


model, backbone = build_improved_model(NUM_AGE_CLASSES)

# Age loss is weighted 1.5x because it is the harder multi-class task
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss={
        "gender_output": "binary_crossentropy",
        "age_output": "categorical_crossentropy"
    },
    loss_weights={
        "gender_output": 1.0,
        "age_output": 1.5
    },
    metrics={
        "gender_output": ["accuracy"],
        "age_output": ["accuracy"]
    }
)

print("\nModel Summary:")
model.summary()


In [ ]:
# ===============================
# CALLBACKS
# ===============================

os.makedirs("/kaggle/working/models", exist_ok=True)

callbacks: list = [
    EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        "/kaggle/working/models/best_model.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

print("Callbacks configured: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint")


## Phase 1: Train Classification Heads (Backbone Frozen)

We first train only the gender and age classification heads while keeping
the MobileNetV2 backbone frozen. This allows the new layers to learn
good initial weights before fine-tuning the backbone.

In [ ]:
# ===============================
# PHASE 1: TRAIN HEADS (BACKBONE FROZEN)
# ===============================

print("=" * 50)
print("PHASE 1: Training classification heads (backbone frozen)")
print("=" * 50)

history_phase1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)


## Phase 2: Fine-Tune Backbone (Unfreezing Last 40 Layers)

Now we unfreeze the last 40 layers of MobileNetV2 and continue training
with a very small learning rate. This allows the backbone to adapt its
learned features specifically for age/gender detection.

In [ ]:
# ===============================
# PHASE 2: FINE-TUNE BACKBONE
# ===============================

# Unfreeze last 40 layers — earlier layers retain generic features
for layer in backbone.layers[-40:]:
    layer.trainable = True

# Recompile at 10x smaller LR to prevent catastrophic forgetting
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE / 10),
    loss={
        "gender_output": "binary_crossentropy",
        "age_output": "categorical_crossentropy"
    },
    loss_weights={
        "gender_output": 1.0,
        "age_output": 1.5
    },
    metrics={
        "gender_output": ["accuracy"],
        "age_output": ["accuracy"]
    }
)

# Tighter callbacks for fine-tuning (lower patience to prevent overfit)
ft_callbacks: list = [
    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-8,
        verbose=1
    ),
    ModelCheckpoint(
        "/kaggle/working/models/best_model_finetuned.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

print("=" * 50)
print("PHASE 2: Fine-tuning backbone (last 40 layers unfrozen)")
print("=" * 50)

history_phase2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=ft_callbacks
)


In [ ]:
# ===============================
# SAVE FINAL MODELS
# ===============================

model.save("/kaggle/working/models/age_gender_model_final.keras")
print("Final model saved to /kaggle/working/models/age_gender_model_final.keras")


In [ ]:
# ===============================
# EVALUATION ON TEST SET
# ===============================

test_gen = FaceDataGenerator(
    paths_test, y_gender_test, y_age_test,
    batch_size=BATCH_SIZE, img_size=IMG_SIZE,
    augment=False, shuffle=False
)

results = model.evaluate(test_gen, verbose=1)

print("\n" + "=" * 50)
print("FINAL TEST RESULTS")
print("=" * 50)
print(f"Total Loss:                  {results[0]:.4f}")
print(f"Gender Loss (BCE):           {results[1]:.4f}")
print(f"Age Loss (CE):               {results[2]:.4f}")
print(f"Gender Accuracy:             {results[3]*100:.2f}%")
print(f"Age Group Accuracy:          {results[4]*100:.2f}%")


In [ ]:
# ===============================
# GENDER DETAILED REPORT
# ===============================

y_gender_pred_all: list[int] = []
y_gender_true_all: list[int] = []

for i in range(len(test_gen)):
    imgs, labels = test_gen[i]
    preds = model.predict(imgs, verbose=0)[0]
    y_gender_pred_all.extend((preds.flatten() > 0.5).astype(int))
    y_gender_true_all.extend(labels["gender_output"])

print("Gender Classification Report:")
print(classification_report(y_gender_true_all, y_gender_pred_all,
                            target_names=["Male", "Female"]))

cm = confusion_matrix(y_gender_true_all, y_gender_pred_all)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Male", "Female"], yticklabels=["Male", "Female"])
plt.title("Gender Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


In [ ]:
# ===============================
# AGE DETAILED REPORT
# ===============================

y_age_pred_all: list[int] = []
y_age_true_all: list[int] = []

for i in range(len(test_gen)):
    imgs, labels = test_gen[i]
    preds = model.predict(imgs, verbose=0)[1]
    y_age_pred_all.extend(np.argmax(preds, axis=1))
    y_age_true_all.extend(np.argmax(labels["age_output"], axis=1))

print("Age Group Classification Report:")
print(classification_report(y_age_true_all, y_age_pred_all,
                            target_names=age_labels_list))

cm_age = confusion_matrix(y_age_true_all, y_age_pred_all)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_age, annot=True, fmt="d", cmap="Oranges",
            xticklabels=age_labels_list, yticklabels=age_labels_list)
plt.title("Age Group Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


In [ ]:
# ===============================
# PREDICTION EXAMPLES
# ===============================

num_examples = 12
sample_indices = np.random.choice(len(paths_test), num_examples, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, idx in enumerate(sample_indices):
    img = cv2.imread(paths_test[idx])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_disp = img.copy()

    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0
    img_input = np.expand_dims(img_resized, axis=0)

    gender_pred, age_pred = model.predict(img_input, verbose=0)
    pred_gender = "Male" if gender_pred[0][0] < 0.5 else "Female"
    pred_age = age_labels_list[np.argmax(age_pred[0])]

    true_gender = "Male" if y_gender_test[idx] == 0 else "Female"
    true_age = age_labels_list[np.argmax(y_age_test[idx])]

    g_ok = (pred_gender == true_gender)
    a_ok = (pred_age == true_age)
    color = "green" if (g_ok and a_ok) else "red"

    axes[i].imshow(img_disp)
    axes[i].axis("off")
    axes[i].set_title(
        f"Pred: {pred_gender}, {pred_age}\nTrue: {true_gender}, {true_age}",
        fontsize=9, color=color
    )

plt.suptitle("Prediction Examples (Green=Correct, Red=Wrong)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# ===============================
# TRAINING HISTORY PLOTS
# ===============================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(history_phase1.history["gender_output_accuracy"], label="Train")
axes[0, 0].plot(history_phase1.history["val_gender_output_accuracy"], label="Val")
axes[0, 0].set_title("Gender Accuracy (Phase 1)")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Accuracy")
axes[0, 0].legend(loc="best")

axes[0, 1].plot(history_phase1.history["age_output_accuracy"], label="Train")
axes[0, 1].plot(history_phase1.history["val_age_output_accuracy"], label="Val")
axes[0, 1].set_title("Age Group Accuracy (Phase 1)")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].legend(loc="best")

axes[1, 0].plot(history_phase2.history["gender_output_accuracy"], label="Train")
axes[1, 0].plot(history_phase2.history["val_gender_output_accuracy"], label="Val")
axes[1, 0].set_title("Gender Accuracy (Phase 2 - Fine-tune)")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Accuracy")
axes[1, 0].legend(loc="best")

axes[1, 1].plot(history_phase2.history["age_output_accuracy"], label="Train")
axes[1, 1].plot(history_phase2.history["val_age_output_accuracy"], label="Val")
axes[1, 1].set_title("Age Group Accuracy (Phase 2 - Fine-tune)")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Accuracy")
axes[1, 1].legend(loc="best")

plt.suptitle("Training History", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# ===============================
# INFERENCE FUNCTION (For New Images)
# ===============================

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)


def predict_age_gender(
    image_path_or_array: str | np.ndarray,
) -> dict[str, str | float]:
    """Predict age group and gender from a face image.

    Accepts either a file path to a JPEG/PNG image or an in-memory
    numpy array (RGB, shape H x W x 3). If a Haar-cascade face is
    detected, the image is cropped to that region before prediction.

    Args:
        image_path_or_array: Either an absolute path to an image file
            (str) or a numpy array in RGB format (H, W, 3), dtype uint8.

    Returns:
        A dictionary with four keys:
            - "gender": "Male" or "Female".
            - "gender_confidence": Float in [0, 1] representing sigmoid
              confidence.
            - "age_group": String label from age_labels_list (e.g. "23-30").
            - "age_confidence": Float in [0, 1] representing softmax
              confidence for the predicted age group.

    Raises:
        FileNotFoundError: If image_path_or_array is a string path but
            the file does not exist or cannot be decoded.
        ValueError: If the input array has an unexpected shape or dtype.
    """
    if isinstance(image_path_or_array, str):
        img = cv2.imread(image_path_or_array)
        if img is None:
            raise FileNotFoundError(
                f"Cannot read image: {image_path_or_array}"
            )
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img = image_path_or_array.copy()

    # Crop to the largest detected face (Haar cascade)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    if len(faces) > 0:
        x, y, w, h = faces[0]
        img = img[y:y+h, x:x+w]

    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0
    img_input = np.expand_dims(img_resized, axis=0)

    gender_pred, age_pred = model.predict(img_input, verbose=0)

    gender = "Male" if gender_pred[0][0] < 0.5 else "Female"
    # Sigmoid: male probability = raw output, female = 1 - raw
    gender_conf = gender_pred[0][0] if gender == "Male" else 1 - gender_pred[0][0]
    age_idx = np.argmax(age_pred[0])
    age_group = age_labels_list[age_idx]
    age_conf = age_pred[0][age_idx]

    return {
        "gender": gender,
        "gender_confidence": float(gender_conf),
        "age_group": age_group,
        "age_confidence": float(age_conf)
    }

print("Inference function ready!")


## Webcam Real-Time Prediction (Optional)

Uncomment the code below to run real-time age & gender prediction
using your webcam. This works in local environments, not on Kaggle.

In [ ]:
# # ===============================
# # WEBCAM REAL-TIME PREDICTION
# # ===============================

# cap = cv2.VideoCapture(0)

# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     result = predict_age_gender(frame)

#     gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
#     faces = face_cascade.detectMultiScale(gray, 1.3, 5)

#     for (x, y, w, h) in faces:
#         label = f"{result['gender']} ({result['gender_confidence']:.0%}), " \
#                 f"Age: {result['age_group']} ({result['age_confidence']:.0%})"
#         cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
#         cv2.putText(frame, label, (x, y-10),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

#     cv2.imshow("Age & Gender Detection", frame)
#     if cv2.waitKey(1) & 0xFF == ord("q"):
#         break

# cap.release()
# cv2.destroyAllWindows()
